# Utiliser des LLM en API pour générer avec une prompt (zero shot)

- des annotations : quels sont les articles qui mentionnent des thématiques IA
- de l'extraction d'information : quelles sont les notions utilisées

Quelques lectures 

- Stuhler, O., Ton, C. D., & Ollion, E. (2025). From Codebooks to Promptbooks: Extracting Information from Text with Generative Large Language Models. Sociological Methods & Research, 54(3), 794-848. https://doi.org/10.1177/00491241251336794 (Original work published 2025)
- https://www.css.cnrs.fr/classification-with-generative-llms-and-api-calls/

## Quelques mots sur le génératif

- Prédiction du next token
- Modèles pré-entrainés
    - des tailles & alignements différents
- Dépendances matérielles fortes

## Quels modèles ?

- vaste question
- équilibre puissance/coût (financier et matériel)
- enjeu de la sécurité des données

De nombreux prestataires

- openai, etc.
- openrouter
- humanum...

## Charger les données

In [1]:
import pandas as pd
df = pd.read_csv("https://raw.githubusercontent.com/pyshs/CUSO2026/refs/heads/main/data/css_openalex_26022026.csv")
df.head()

,id,type,primary_location,title,abstract_inverted_index,publication_year,publication_date,open_access,relevance_score,abstract,journal
0,https://openalex.org/W2159397589,article,"{'id': 'doi:10.1126/science.1167742', 'is_oa':...",Computational Social Science,"{'A': [0], 'field': [1], 'is': [2], 'emerging'...",2009.0,2009-02-06,"{'is_oa': True, 'oa_status': 'green', 'oa_url'...",1360.35770,A field is emerging that leverages the capacit...,Science
1,https://openalex.org/W2070907364,article,"{'id': 'doi:10.1140/epjst/e2012-01697-8', 'is_...",Manifesto of computational social science,NaN,2012.0,2012-11-01,"{'is_oa': True, 'oa_status': 'hybrid', 'oa_url...",497.82666,NaN,The European Physical Journal Special Topics
2,https://openalex.org/W3081158114,article,"{'id': 'doi:10.1126/science.aaz8170', 'is_oa':...",Computational social science: Obstacles and op...,"{'Data': [0], 'sharing,': [1], 'research': [2]...",2020.0,2020-08-28,"{'is_oa': True, 'oa_status': 'green', 'oa_url'...",438.53986,"Data sharing, research ethics, and incentives ...",Science
3,https://openalex.org/W3022499311,article,{'id': 'doi:10.1146/annurev-soc-121919-054621'...,Computational Social Science and Sociology,"{'The': [0], 'integration': [1], 'of': [2, 16,...",2020.0,2020-04-28,"{'is_oa': True, 'oa_status': 'hybrid', 'oa_url...",413.03424,The integration of social science with compute...,Annual Review of Sociology
4,https://openalex.org/W3174174150,article,"{'id': 'doi:10.1038/s41586-021-03659-0', 'is_o...",Integrating explanation and prediction in comp...,NaN,2021.0,2021-06-30,"{'is_oa': False, 'oa_status': 'closed', 'oa_ur...",408.08980,NaN,Nature


## Charger une clé

In [4]:
import yaml
with open("config.yaml", "r") as f:
    config = yaml.safe_load(f)
api_key = config["api_key"]

## Utiliser openrouter et l'API openapi

Quickstart : https://platform.openai.com/docs/quickstart?api-mode=chat&language=python

In [6]:
#!pip install openai

In [7]:
from openai import OpenAI

api_url = "https://openrouter.ai/api/v1" # endpoint
 
client = OpenAI(
  base_url=api_url,
  api_key=api_key,
)

response = client.chat.completions.create(
    model="meta-llama/llama-3.3-70b-instruct",
    messages=[{"role": "user", 
               "content": "Est-ce que Python est un langage adapté pour les sciences sociales"}]
)

response

ChatCompletion(id='gen-1774004876-OxxXqprSQB7Vjo7U1ddl', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content="Oui, Python est un langage de programmation très adapté pour les sciences sociales. Voici quelques raisons pour lesquelles :\n\n1. **Analyse de données** : Python offre une grande variété de bibliothèques et d'outils pour l'analyse de données, telles que Pandas, NumPy, Matplotlib et Scikit-learn. Ces outils permettent de manipuler, de visualiser et de modéliser des données de manière efficace.\n2. **Traitement de données textuelles** : Python est particulièrement bien adapté pour le traitement de données textuelles, grâce à des bibliothèques comme NLTK (Natural Language Toolkit), SpaCy et Gensim. Ces outils permettent de nettoyer, de tokeniser, de poser des étiquettes et d'analyser des textes de manière automatisée.\n3. **Modélisation et simulation** : Python offre des bibliothèques comme Scipy et PyMC3 pour la modélisation et la 

Choisir un modèle

df["abstract"].dropna().iloc[1]Construire la prompt

In [9]:
df["abstract"].dropna().iloc[2]

'The integration of social science with computer science and engineering fields has produced a new area of study: computational social science. This field applies computational methods to novel sources of digital data such as social media, administrative records, and historical archives to develop theories of human behavior. We review the evolution of this field within sociology via bibliometric analysis and in-depth analysis of the following subfields where this new work is appearing most rapidly: (<i>a</i>) social network analysis and group formation; (<i>b</i>) collective behavior and political sociology; (<i>c</i>) the sociology of knowledge; (<i>d</i>) cultural sociology, social psychology, and emotions; (<i>e</i>) the production of culture; (<i>f</i>) economic sociology and organizations; and (<i>g</i>) demography and population studies. Our review reveals that sociologists are not only at the center of cutting-edge research that addresses longstanding questions about human behav

In [15]:
messages = [
  {
    "role": "system",
    "content": "You are an efficient research assistant helping with text annotation."
  },
  {
    "role": "user",
    "content": """
    Annotate this following text if its topic is about AI or algorithms. AI regroups all the topics connected with
    machine learning, deep learning and models.
    
    If so, returns AI, and otherwise, return NOT AI.
    
    Only return AI or NOT AI, nothing else.
    """
      + df["abstract"].dropna().iloc[2]
  }
]

In [17]:
df["abstract"].dropna().iloc[2]

'The integration of social science with computer science and engineering fields has produced a new area of study: computational social science. This field applies computational methods to novel sources of digital data such as social media, administrative records, and historical archives to develop theories of human behavior. We review the evolution of this field within sociology via bibliometric analysis and in-depth analysis of the following subfields where this new work is appearing most rapidly: (<i>a</i>) social network analysis and group formation; (<i>b</i>) collective behavior and political sociology; (<i>c</i>) the sociology of knowledge; (<i>d</i>) cultural sociology, social psychology, and emotions; (<i>e</i>) the production of culture; (<i>f</i>) economic sociology and organizations; and (<i>g</i>) demography and population studies. Our review reveals that sociologists are not only at the center of cutting-edge research that addresses longstanding questions about human behav

In [16]:
#!pip install openai
from openai import OpenAI
api_url = "https://openrouter.ai/api/v1"
 
client = OpenAI(
  base_url=api_url,
  api_key=api_key,
)

response = client.chat.completions.create(
    model="meta-llama/llama-3.3-70b-instruct",
    messages=messages
)
response.choices[0].message.content

'NOT AI'

### Application Faire une boucle sur une liste de documents : évaluer si un abstract porte ou pas sur l'IA

- Tester avec des exemples (few shot)
- Tester avec du raisonnement (chaîne of thought)

In [24]:
from openai import OpenAI
api_url = "https://openrouter.ai/api/v1"

client = OpenAI(
  base_url=api_url,
  api_key=api_key,
)

annotations = []

for element in df["abstract"][0:20]:
    print("Start annotation element")
    if pd.isna(element) or len(element)==0:
        annotations.append("empty")
        continue
    if len(element)>10000:
        annotations.append("skip")
        continue

        
    messages = [
      {
        "role": "system",
        "content": "You are an efficient research assistant helping with text annotation."
      },
      {
        "role": "user",
        "content": """
        Annotate this following text if its topic is about AI or algorithms. AI regroups all the topics connected with
        machine learning, deep learning and models.

        If so, returns AI, and otherwise, return NOT AI.

        Only return AI or NOT AI, nothing else.
        """
          + element
      }]
        
    response = client.chat.completions.create(
        model="meta-llama/llama-3.3-70b-instruct",
        messages=messages
    )
    content = response.choices[0].message.content
    annotations.append(content)

Start annotation element
Start annotation element
Start annotation element
Start annotation element
Start annotation element
Start annotation element
Start annotation element
Start annotation element
Start annotation element
Start annotation element
Start annotation element
Start annotation element
Start annotation element
Start annotation element
Start annotation element
Start annotation element
Start annotation element
Start annotation element
Start annotation element
Start annotation element


In [25]:
annotations

['NOT AI',
 'empty',
 'NOT AI',
 'NOT AI',
 'empty',
 'AI',
 'empty',
 'NOT AI',
 'empty',
 'AI',
 'empty',
 'NOT AI',
 'empty',
 'AI',
 'empty',
 'NOT AI',
 'empty',
 'empty',
 'empty',
 'empty']

In [27]:
ground_truth = ['AI',
 'empty',
 'NOT AI',
 'NOT AI',
 'empty',
 'AI',
 'empty',
 'NOT AI',
 'empty',
 'AI',
 'empty',
 'AI',
 'empty',
 'NOT AI',
 'empty',
 'NOT AI',
 'empty',
 'empty',
 'empty',
 'empty']

In [32]:
from sklearn.metrics import classification_report

print(classification_report(annotations, ground_truth))

              precision    recall  f1-score   support

          AI       0.50      0.67      0.57         3
      NOT AI       0.80      0.67      0.73         6
       empty       1.00      1.00      1.00        11

    accuracy                           0.85        20
   macro avg       0.77      0.78      0.77        20
weighted avg       0.86      0.85      0.85        20



In [36]:
from sklearn.metrics import confusion_matrix

pd.DataFrame(confusion_matrix(annotations, ground_truth), 
             columns = ["AI", "NOT AI", "empty"],
            index = ["AI", "NOT AI", "empty"])

,AI,NOT AI,empty
AI,2,1,0
NOT AI,2,4,0
empty,0,0,11


## Evaluation

Comment évaluer la qualité d'un codage ? 

- La nécessité d'un gold standard
- Le calcul de métriques `from sklearn.metrics import classification_report`

### Application : construire un jeu de données d'évaluation

- Utiliser des outils permettant l'interaction humaine
- Toujours de l'ambiguité : accord inter-annotateur (alpha de Chronbach, autre)

### Mesurer la qualité d'une évaluation

Différentes métriques

- précision
- recall
- f1

In [ ]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report

## Application : extraire de l'information

Sur les articles identifiés comme AI-related, extraire les termes liés pour les analyser

- Construire une prompt
- Tester sur plusieurs textes
- Gérer la structure des données